<a href="https://colab.research.google.com/github/Samriddhi28-17/Text-to-Audio-withPython/blob/Samriddhi28-17/Text_to_audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install streamlit PyMuPDF gTTS
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.1/24.1 MB 75.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 98.2/98.2 kB 9.6 MB/s eta 0:00:00
  Attempting uninstall: click
    Found existing installation: click 8.2.1
    Uninstalling click-8.2.1:
      Successfully uninstalled click-8.2.1


In [ ]:
import fitz

In [ ]:
from gtts import gTTS
import os

In [ ]:
from google.colab import files
from IPython.display import display, Audio

os.makedirs('pdf_converter_files', exist_ok=True)
os.chdir('pdf_converter_files')

In [ ]:
def extract_text_from_pdf(pdf_path, pages=None):
    text = ""
    try:
        doc = fitz.open(pdf_path)
        if pages is None:
            pages_to_read = range(len(doc))
        else:
            pages_to_read = [p - 1 for p in pages if 0 < p <= len(doc)]
        for page_num in pages_to_read:
            page = doc.load_page(page_num)
            text += page.get_text()
        return text
    except Exception as e:
        return f"An error occurred: {e}"

In [ ]:
def convert_text_to_audio(text, output_file, lang='en'):
    try:
        tts = gTTS(text=text, lang=lang, slow=False)
        tts.save(output_file)
        print(f"Audio saved to {output_file}")
    except Exception as e:
        print(f"An error occurred during audio conversion: {e}")

In [ ]:
def main():
    print("Please upload your PDF file.")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded. Please try again.")
        return

    pdf_filename = list(uploaded.keys())[0]

    # Use a dropdown for a simple whole book/custom pages choice
    option = 'Full book' #@param ["Full book", "Custom pages"]

    pages_input = '1, 3-5, 8' #@param {type:"string"}

    output_filename = 'audiobook.mp3' #@param {type:"string"}

    if option == 'Full book':
        pages_to_read = None
        print(f"Converting the entire book: {pdf_filename}")
    else:
        # Parse the custom pages input string
        pages_to_read = []
        try:
            for part in pages_input.replace(" ", "").split(','):
                if '-' in part:
                    start, end = map(int, part.split('-'))
                    pages_to_read.extend(range(start, end + 1))
                else:
                    pages_to_read.append(int(part))
        except ValueError:
            print("Invalid page input. Please use a format like '1, 3-5, 8'.")
            return

        print(f"Converting pages {pages_to_read} from {pdf_filename}")

    extracted_text = extract_text_from_pdf(pdf_filename, pages=pages_to_read)

    if "An error occurred" in extracted_text:
        print(extracted_text)
        return

    if not extracted_text.strip():
        print("No text was extracted from the specified pages. The PDF might be an image-based scan.")
        return

    # Convert the extracted text to audio
    convert_text_to_audio(extracted_text, output_filename)

    # Offer the user to play the audio and download it
    print("\nAudio conversion complete!")
    print("You can play the audio below or download it.")
    display(Audio(output_filename, autoplay=False))
    files.download(output_filename)

In [ ]:
if __name__ == "__main__":
    main()

Please upload your PDF file.


No file uploaded. Please try again.
